In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, txt, tokenizer, context_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - context_length, stride):
            input_chunk = token_ids[i:i + context_length]
            target_chunk = token_ids[i + 1:i + context_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [2]:
import tiktoken

def create_dataloader(txt, batch_size=4, context_length=256, stride=128):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDataset(txt, tokenizer, context_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    return dataloader

# Test with a small text
text = "Every effort moves you forward. Every effort moves you forward. " * 100
dl = create_dataloader(text, batch_size=2, context_length=16, stride=16)

batch = next(iter(dl))
inputs, targets = batch
print("Inputs shape:", inputs.shape)
print("Targets shape:", targets.shape)
print("Inputs[0]:", inputs[0])
print("Targets[0]:", targets[0])

Inputs shape: torch.Size([2, 16])
Targets shape: torch.Size([2, 16])
Inputs[0]: tensor([2651,   13, 3887, 3626, 6100,  345, 2651,   13, 3887, 3626, 6100,  345,
        2651,   13, 3887, 3626])
Targets[0]: tensor([  13, 3887, 3626, 6100,  345, 2651,   13, 3887, 3626, 6100,  345, 2651,
          13, 3887, 3626, 6100])


In [3]:
tokenizer = tiktoken.get_encoding("gpt2")
print("Input text:", tokenizer.decode(inputs[0].tolist()))
print("Target text:", tokenizer.decode(targets[0].tolist()))

Input text:  forward. Every effort moves you forward. Every effort moves you forward. Every effort
Target text: . Every effort moves you forward. Every effort moves you forward. Every effort moves


In [ ]:
torch.manual_seed(123)
device = torch.device("cuda")

model = GPTModel(
    vocab_size=50257,
    emb_dim=128,
    context_length=256,
    num_heads=4,
    num_layers=4,
    dropout=0.1
).to(device)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {next(model.parameters()).device}")